# Summary Generator for large document using transformer based model
BART- Bidirectional Auto Regressor transformers

*   It reads from left and right thats why bidirectional.
*   Limitation: Only 1024 words it can take and removes(truncates) the rest
*  Gives token Id after tokenization like <224545>






Since it can take only 1024 words we need to break the documents into many chunks such as chunk1(one chunk 500 words), chunk2....
In the end summary will be generated for each chunk and it will be combined to form one summary.

In [ ]:
!pip install -q torch torchvision torchaudio transformers sentencepiece accelerate

In [ ]:
import torch
print(torch.__version__)

2.11.0+cu130


Load pre trained Transformer Model

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration
import re

model_name="facebook/bart-large-cnn"
tokenizer=BartTokenizer.from_pretrained(model_name)
model=BartForConditionalGeneration.from_pretrained(model_name)
print("Model loaded successfully!")

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded successfully!


Text PreProcessing

In [ ]:
def preprocess_text(text):
  text=re.sub(r'\s+',' ',text)
  text=text.strip()
  return text

Sample Large Document

In [ ]:
document="""
Agentic AI is an artificial intelligence system that can accomplish a specific goal with limited supervision.
It consists of AI agents—machine learning models that mimic human decision-making to solve problems in real time.
In a multiagent system, each agent performs a specific subtask required to reach the goal and their efforts are coordinated through AI orchestration.
"""
document= document * 50

In [ ]:
document=preprocess_text(document)
print("preprocessing completed")

preprocessing completed


tokenize the doc

In [ ]:
inputs= tokenizer(document, return_tensors="pt", max_length=1024, truncation=True)
print("Tokenization completed")

Tokenization completed


Generate summary

In [ ]:
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=100,
    min_length=30,
    num_beams=4,
    early_stopping=True
)

Decode the summary

In [ ]:
summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Generated Summary:")
print(summary)

Generated Summary:
Agentic AI is an artificial intelligence system that can accomplish a specific goal with limited supervision. It consists of AI agents that mimic human decision-making to solve problems in real time. In a multiagent system, each agent performs a specific subtask required to reach the goal.


Transformer models have input limits. Therefore, we split long documents into chunks.


Create Chunking Function

In [ ]:
def split_text(text, chunk_size=500):
  words=text.split()
  chunks=[]
  for i in range(0, len(words), chunk_size):
    chunk=" ".join(words[i:i+chunk_size])
    chunks.append(chunk)
  return chunks


Summarize Large Document

In [ ]:
def summarize_large_document(text):
  chunks=split_text(text)
  summaries=[]
  for chunk in chunks:
    inputs =tokenizer(chunk, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids=model.generate(inputs["input_ids"], max_length=80, min_length=20,num_beams=4)
    summary=tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    summaries.append(summary)
  final_summary=" ".join(summaries)
  return final_summary

In [ ]:
final_summary=summarize_large_document(document)
print("Final Summary:\n")
print(final_summary)



Final Summary:

Agentic AI is an artificial intelligence system that can accomplish a specific goal with limited supervision. It consists of AI agents—machine learning models that mimic human decision-making to solve problems in real time. In a multiagent system, each agent performs a specific subtask required to reach the goal. Agentic AI is an artificial intelligence system that can accomplish a specific goal with limited supervision. It consists of AI agents that mimic human decision-making to solve problems in real time. In a multiagent system, each agent performs a specific subtask required to reach the goal. Agentic AI is an artificial intelligence system that can accomplish a specific goal with limited supervision. It consists of AI agents that mimic human decision-making to solve problems in real time. In a multiagent system, each agent performs a specific subtask required to reach the goal. Agentic AI is an artificial intelligence system that can accomplish a specific goal wit

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration
import re
model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

text = """
Artificial intelligence is transforming many industries.
It is used in healthcare, education, finance and transportation.
AI systems can analyze large amounts of data and assist humans
in making decisions.
Artificial intelligence is transforming many industries.
It is used in healthcare, education, finance and transportation.
AI systems can analyze large amounts of data and assist humans
in making decisions.
Artificial intelligence is transforming many industries.
It is used in healthcare, education, finance and transportation.
AI systems can analyze large amounts of data and assist humans
in making decisions.
"""
def preprocess_text(text):
  text=re.sub(r'\s+',' ',text)
  text=text.strip()
  return text

text=preprocess_text(text)

inputs = tokenizer(
    text,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=100,
    min_length=30,
    num_beams=4
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Summary:")
print(summary)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Summary:
Artificial intelligence is transforming many industries. It is used in healthcare, education, finance and transportation. AI systems can analyze large amounts of data and assist humans in making decisions.


In [ ]:
import gradio as gr

def summarize_file(file):

    with open(file, "r", encoding="utf-8") as f:
        text = f.read()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=100,
        min_length=30,
        num_beams=4
    )

    summary = tokenizer.decode(
        summary_ids[0], skip_special_tokens=True)

    return summary

interface=gr.Interface(fn=summarize_file,
                       inputs=gr.File(type='filepath',
                                      file_types=['.txt']),
                       outputs=gr.Textbox())

interface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://31eb44d26447035b87.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def preprocess_text(text):
  text=re.sub(r'\s+',' ',text)
  text=text.strip()
  return text

text=preprocess_text(text)

def split_text(text, chunk_size=500):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))

    return chunks


def summarize_large_document(text):
    chunks = split_text(text)
    summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            max_length=1024,
            truncation=True
        )

        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=80,
            min_length=20,
            num_beams=4
        )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        summaries.append(summary)

    return " ".join(summaries)

In [ ]:
final_summary = summarize_large_document(text)
print(final_summary)

Artificial intelligence is transforming many industries. It is used in healthcare, education, finance and transportation. It can analyze large amounts of data and assist humans in making decisions.


In [ ]:
import gradio as gr
from transformers import BartTokenizer, BartForConditionalGeneration
import re

# Load model
model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)


# Preprocess text
def preprocess_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text


# Split large text into chunks
def split_text(text, chunk_size=500):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))

    return chunks


# Summarize large document
def summarize_large_document(file):

    # Read uploaded TXT file
    with open(file, "r", encoding="utf-8") as f:
        text = f.read()

    # Preprocess
    text = preprocess_text(text)

    # Split into chunks
    chunks = split_text(text)

    summaries = []

    # Summarize each chunk
    for chunk in chunks:

        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            max_length=1024,
            truncation=True
        )

        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=80,
            min_length=20,
            num_beams=4
        )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        summaries.append(summary)

    # Combine all summaries
    return " ".join(summaries)


# Gradio interface
app = gr.Interface(
    fn=summarize_large_document,
    inputs=gr.File(
        type="filepath",
        file_types=[".txt"],
        label="Upload TXT File"
    ),
    outputs=gr.Textbox(
        label="Final Summary"
    ),
    title="Large Document Summarization using BART",
    description="Upload a TXT file to generate a summary"
)

app.launch(share=True)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8778da5f0ea394ca06.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
